In [ ]:
import pandas as pd

In [ ]:
from google.cloud import bigquery

client = bigquery.Client()
query = """
    SELECT * FROM `numeric-advice-452700-j9.neo_bank_.retencion_conversion`
"""
df_retencion_conversion = client.query(query).to_dataframe()

df_retencion_conversion

In [ ]:
import pandas as pd

# Suponiendo que ya cargaste el DataFrame de la tabla retencion_conversion en df_retencion_conversion
# por ejemplo:
# df_retencion_conversion = client.query("SELECT * FROM `neo_bank_.retencion_conversion`").to_dataframe()

# Convertir fechas a datetime (si no lo están)
df_retencion_conversion['create_date_user'] = pd.to_datetime(df_retencion_conversion['create_date_user'], errors='coerce')
df_retencion_conversion['first_transaction_date'] = pd.to_datetime(df_retencion_conversion['first_transaction_date'], errors='coerce')

# Calcular días a primera transacción
df_retencion_conversion['dias_a_primera_txn'] = (
    (df_retencion_conversion['first_transaction_date'] - df_retencion_conversion['create_date_user']).dt.days
)

# Mostrar primeros registros para revisar
df_retencion_conversion


In [ ]:
import plotly.express as px

# Histograma de días a la primera transacción
fig_dias = px.histogram(
    df_retencion_conversion.dropna(subset=['dias_a_primera_txn']),
    x='dias_a_primera_txn',
    nbins=200,
    labels={'dias_a_primera_txn': 'Días hasta la primera transacción'},
    title='Distribución de días hasta la primera transacción',
    color_discrete_sequence=['#3BAEDA']
)

fig_dias.update_layout(
    xaxis_title='Días a primera transacción',
    yaxis_title='Número de usuarios',
    bargap=0.1
)

fig_dias.show()


# % de usuarios que convierten en distintos rangos de días

In [ ]:
import pandas as pd
import plotly.express as px

# 1) Copia para no modificar el original
df_txn = df_retencion_conversion.dropna(subset=['first_transaction_date']).copy()

# 2) Convertir fechas a datetime sin timezone
df_txn['first_transaction_date'] = pd.to_datetime(df_txn['first_transaction_date']).dt.tz_localize(None)
df_txn['create_date_user'] = pd.to_datetime(df_txn['create_date_user']).dt.tz_localize(None)

# 3) Calcular días a la primera transacción
df_txn['dias_a_primera_txn'] = (df_txn['first_transaction_date'] - df_txn['create_date_user']).dt.days

# 4) Conservar solo el primer registro de cada usuario (por si hubiera duplicados)
df_first_txn = df_txn.sort_values('dias_a_primera_txn').drop_duplicates(subset='user_id', keep='first')

# 5) Calcular columnas booleanas para los distintos rangos
df_first_txn['converted_1d'] = df_first_txn['dias_a_primera_txn'] <= 1
df_first_txn['converted_7d'] = df_first_txn['dias_a_primera_txn'] <= 7
df_first_txn['converted_30d'] = df_first_txn['dias_a_primera_txn'] <= 30
df_first_txn['converted_60d'] = df_first_txn['dias_a_primera_txn'] <= 60
df_first_txn['converted_90d'] = df_first_txn['dias_a_primera_txn'] <= 90
df_first_txn['converted_120d'] = df_first_txn['dias_a_primera_txn'] <= 120
df_first_txn['converted_mayor_120d'] = df_first_txn['dias_a_primera_txn'] > 120

# 6) Calcular el total de usuarios registrados
total_users = df_retencion_conversion['user_id'].nunique()

# 7) Calcular tasas de conversión por rango
conversion_data = {
    'Periodo': ['1 día', '7 días', '30 días', '60 días', '90 días', '120 días', 'Mayor a 120 días'],
    'Conversión (%)': [
        df_first_txn['converted_1d'].sum() / total_users * 100,
        df_first_txn['converted_7d'].sum() / total_users * 100,
        df_first_txn['converted_30d'].sum() / total_users * 100,
        df_first_txn['converted_60d'].sum() / total_users * 100,
        df_first_txn['converted_90d'].sum() / total_users * 100,
        df_first_txn['converted_120d'].sum() / total_users * 100,
        df_first_txn['converted_mayor_120d'].sum() / total_users * 100
    ]
}

# 8) Graficar el histograma
fig_histograma_conversion = px.bar(
    conversion_data,
    x='Periodo',
    y='Conversión (%)',
    text='Conversión (%)',
    color='Periodo',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'Periodo': 'Rango de días', 'Conversión (%)': 'Conversión (%)'},
    title='% de usuarios que convierten en distintos rangos de días'
)

fig_histograma_conversion.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_histograma_conversion.update_layout(yaxis_range=[0, 100])

fig_histograma_conversion.show()
